In [ ]:
import os
import shutil
from typing import List, Set, Optional
from pathlib import Path


def get_folder_size(folder_path: str) -> float:
    """计算文件夹大小（MB）"""
    total = 0
    for dirpath, dirnames, filenames in os.walk(folder_path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            if os.path.exists(fp):
                total += os.path.getsize(fp)
    return total / (1024 * 1024)  # MB


def list_scenario_contents(
    base_path: str,
    max_scenarios: int = 3
) -> None:
    """
    列出输出文件夹的结构和内容详情。
    
    Parameters
    ----------
    base_path : str
        输出文件夹根路径，如 'output/chessboardCarrierReceiverCollab'
    max_scenarios : int
        显示详情的场景数量
    """
    if not os.path.exists(base_path):
        print(f"路径不存在: {base_path}")
        return
    
    all_items = os.listdir(base_path)
    folders = [f for f in all_items if os.path.isdir(os.path.join(base_path, f)) and not f.startswith('.')]
    
    print(f"=" * 60)
    print(f"基础路径: {base_path}")
    print(f"子文件夹总数: {len(folders)}")
    print(f"=" * 60)
    
    # 计算总大小
    total_size = get_folder_size(base_path)
    print(f"总大小: {total_size:.2f} MB ({total_size/1024:.2f} GB)")
    print()
    
    # 显示部分场景的详情
    for i, folder in enumerate(sorted(folders)[:max_scenarios]):
        folder_path = os.path.join(base_path, folder)
        folder_size = get_folder_size(folder_path)
        print(f"\n[{i+1}] {folder} ({folder_size:.2f} MB)")
        print("-" * 50)
        
        contents = os.listdir(folder_path)
        files = [f for f in contents if os.path.isfile(os.path.join(folder_path, f))]
        dirs = [f for f in contents if os.path.isdir(os.path.join(folder_path, f))]
        
        print(f"  文件 ({len(files)}):")
        for f in sorted(files)[:15]:
            fpath = os.path.join(folder_path, f)
            fsize = os.path.getsize(fpath) / 1024  # KB
            print(f"    - {f} ({fsize:.1f} KB)")
        if len(files) > 15:
            print(f"    ... 还有 {len(files)-15} 个文件")
        
        print(f"  文件夹 ({len(dirs)}):")
        for d in sorted(dirs):
            dpath = os.path.join(folder_path, d)
            dsize = get_folder_size(dpath)
            if d == 'ITERS':
                iters_contents = os.listdir(dpath)
                iter_folders = [x for x in iters_contents if x.startswith('it.')]
                print(f"    - {d}/ ({dsize:.2f} MB, {len(iter_folders)} iterations: {sorted(iter_folders)[:5]}...)")
            else:
                print(f"    - {d}/ ({dsize:.2f} MB)")


def clean_output_folders(
    base_path: str,
    files_to_keep: List[str] = None,
    patterns_to_keep: List[str] = None,
    folders_to_keep: List[str] = None,
    iters_to_keep: List[int] = None,
    dry_run: bool = True,
    verbose: bool = True
) -> dict:
    """
    清理输出文件夹，只保留指定的文件和文件夹。
    
    Parameters
    ----------
    base_path : str
        输出文件夹根路径
    files_to_keep : List[str], optional
        要保留的精确文件名列表，如 ['output_carriers.xml.gz', 'output_events.xml.gz']
    patterns_to_keep : List[str], optional
        要保留的文件名模式（包含匹配），如 ['carrier_scores', '.png']
    folders_to_keep : List[str], optional
        要保留的文件夹名（除ITERS外），如 ['analysis']
    iters_to_keep : List[int], optional
        要保留的iteration编号，如 [0, 50] 表示保留 it.0 和 it.50
    dry_run : bool
        如果True，只显示会删除什么，不实际删除
    verbose : bool
        是否显示详细信息
        
    Returns
    -------
    dict : 包含统计信息的字典
    """
    # 默认值
    if files_to_keep is None:
        files_to_keep = [
            'output_carriers.xml.gz',
            'output_events.xml.gz',
            'carriers.xml.gz',
            'receivers.xml.gz',
            'receiver_stats.csv',
            'carrier_scores.txt',
            'receiver_scores.txt',
        ]
    
    if patterns_to_keep is None:
        patterns_to_keep = [
            'carrier_scores.png',
            'receiver_scores.png',
        ]
    
    if folders_to_keep is None:
        folders_to_keep = []  # 除ITERS外，默认不保留其他文件夹
    
    if iters_to_keep is None:
        iters_to_keep = [0, 50]  # 默认只保留第一个和最后一个iteration
    
    # 确保ITERS始终在保留列表中（但会清理其内部）
    if 'ITERS' not in folders_to_keep:
        folders_to_keep = folders_to_keep + ['ITERS']
    
    iters_to_keep_names = {f'it.{i}' for i in iters_to_keep}
    files_to_keep_set = set(files_to_keep)
    
    def should_keep_file(filename: str) -> bool:
        """判断文件是否应该保留"""
        if filename in files_to_keep_set:
            return True
        for pattern in patterns_to_keep:
            if pattern in filename:
                return True
        return False
    
    stats = {
        'scenarios_processed': 0,
        'files_deleted': 0,
        'folders_deleted': 0,
        'bytes_freed': 0,
        'files_kept': 0,
        'errors': []
    }
    
    if not os.path.exists(base_path):
        print(f"路径不存在: {base_path}")
        return stats
    
    # 获取所有场景文件夹
    all_items = os.listdir(base_path)
    scenario_folders = [f for f in all_items 
                       if os.path.isdir(os.path.join(base_path, f)) and not f.startswith('.')]
    
    mode_str = "[DRY RUN] " if dry_run else ""
    print(f"{mode_str}开始清理 {base_path}")
    print(f"保留文件: {files_to_keep}")
    print(f"保留模式: {patterns_to_keep}")
    print(f"保留文件夹: {folders_to_keep}")
    print(f"保留iterations: {iters_to_keep}")
    print(f"场景总数: {len(scenario_folders)}")
    print("=" * 60)
    
    for scenario in sorted(scenario_folders):
        scenario_path = os.path.join(base_path, scenario)
        stats['scenarios_processed'] += 1
        
        if verbose:
            print(f"\n处理: {scenario}")
        
        # 处理场景根目录的文件
        for item in os.listdir(scenario_path):
            item_path = os.path.join(scenario_path, item)
            
            if os.path.isfile(item_path):
                if should_keep_file(item):
                    stats['files_kept'] += 1
                    if verbose:
                        print(f"  ✓ 保留文件: {item}")
                else:
                    file_size = os.path.getsize(item_path)
                    stats['bytes_freed'] += file_size
                    stats['files_deleted'] += 1
                    if verbose:
                        print(f"  ✗ 删除文件: {item} ({file_size/1024:.1f} KB)")
                    if not dry_run:
                        try:
                            os.remove(item_path)
                        except Exception as e:
                            stats['errors'].append(f"删除文件失败 {item_path}: {e}")
                            
            elif os.path.isdir(item_path):
                if item == 'ITERS':
                    # 特殊处理ITERS文件夹
                    iters_path = item_path
                    if os.path.exists(iters_path):
                        for iter_folder in os.listdir(iters_path):
                            iter_path = os.path.join(iters_path, iter_folder)
                            if os.path.isdir(iter_path):
                                if iter_folder in iters_to_keep_names:
                                    if verbose:
                                        print(f"  ✓ 保留iteration: {iter_folder}")
                                else:
                                    folder_size = get_folder_size(iter_path) * 1024 * 1024  # bytes
                                    stats['bytes_freed'] += folder_size
                                    stats['folders_deleted'] += 1
                                    if verbose:
                                        print(f"  ✗ 删除iteration: {iter_folder} ({folder_size/1024/1024:.2f} MB)")
                                    if not dry_run:
                                        try:
                                            shutil.rmtree(iter_path)
                                        except Exception as e:
                                            stats['errors'].append(f"删除文件夹失败 {iter_path}: {e}")
                                            
                elif item in folders_to_keep:
                    if verbose:
                        print(f"  ✓ 保留文件夹: {item}/")
                else:
                    folder_size = get_folder_size(item_path) * 1024 * 1024  # bytes
                    stats['bytes_freed'] += folder_size
                    stats['folders_deleted'] += 1
                    if verbose:
                        print(f"  ✗ 删除文件夹: {item}/ ({folder_size/1024/1024:.2f} MB)")
                    if not dry_run:
                        try:
                            shutil.rmtree(item_path)
                        except Exception as e:
                            stats['errors'].append(f"删除文件夹失败 {item_path}: {e}")
    
    # 打印统计
    print("\n" + "=" * 60)
    print(f"{mode_str}清理统计:")
    print(f"  处理场景数: {stats['scenarios_processed']}")
    print(f"  保留文件数: {stats['files_kept']}")
    print(f"  删除文件数: {stats['files_deleted']}")
    print(f"  删除文件夹数: {stats['folders_deleted']}")
    print(f"  释放空间: {stats['bytes_freed']/1024/1024:.2f} MB ({stats['bytes_freed']/1024/1024/1024:.2f} GB)")
    
    if stats['errors']:
        print(f"\n错误 ({len(stats['errors'])}):")
        for err in stats['errors'][:10]:
            print(f"  - {err}")
    
    if dry_run:
        print(f"\n⚠️  这是 DRY RUN 模式，没有实际删除任何文件。")
        print(f"    设置 dry_run=False 来实际执行删除操作。")
    
    return stats

In [ ]:
# 配置路径
base_path = 'output/chessboardCarrierReceiverCollabCorrect'


# 先查看文件夹结构
list_scenario_contents(base_path, max_scenarios=2)

In [ ]:
# =============================================================================
# DRY RUN: 预览将要删除的内容（不实际删除）
# =============================================================================

# 自定义保留的文件和文件夹
my_files_to_keep = [
    'output_carriers.xml.gz',
    'output_allVehicles.xml.gz',
    'output_config_reduced.xml',
    'output_events.xml.gz', 
    'receivers.xml.gz',
    'receiver_stats.csv',
    'carrier_scores.txt',
    'receiver_scores.txt',
    'output_receiverInTourPlacement.csv.gz'
    # 'carrier_scores.png',
    # 'receiver_scores.png',
]

my_patterns_to_keep = [
    'carrier_scores.png',
    'receiver_scores.png',
]

my_folders_to_keep = [
    'analysis'
    ]  # 除ITERS外的其他文件夹，如 ['analysis']

my_iters_to_keep = [0, 30]  # 保留的iteration编号



In [ ]:
# DRY RUN - 只预览，不删除
stats = clean_output_folders(
    base_path=base_path,
    files_to_keep=my_files_to_keep,
    patterns_to_keep=my_patterns_to_keep,
    folders_to_keep=my_folders_to_keep,
    iters_to_keep=my_iters_to_keep,
    dry_run=True,  # ⚠️ 设为 False 才会真正删除
    verbose=True   # 设为 True 查看每个文件的详情
)

In [ ]:
# =============================================================================
# ⚠️ 实际删除 - 请确认 DRY RUN 结果后再运行此单元格！
# =============================================================================

# 取消下面的注释来执行真正的删除操作
stats = clean_output_folders(
    base_path=base_path,
    files_to_keep=my_files_to_keep,
    patterns_to_keep=my_patterns_to_keep,
    folders_to_keep=my_folders_to_keep,
    iters_to_keep=my_iters_to_keep,
    dry_run=False,  # ⚠️ 这会真正删除文件！
    verbose=False
)

In [ ]:
# 删除后检查剩余空间
list_scenario_contents(base_path, max_scenarios=2)